#Task 1: Build a Robust NLP Preprocessing Engine (Advanced)


## Task 1: Conceptual Understanding

### 1. Difference between "Love" and "love"

In NLP, "Love" and "love" are considered different words because of case sensitivity. This can unnecessarily increase the vocabulary size and create confusion for the model. To avoid this, we usually convert all text into lowercase so that both are treated as the same word.

---

### 2. What happens if stopwords are not removed?

If stopwords are not removed, the dataset contains a lot of unnecessary words like "is", "the", "and", etc. These words do not add much meaning but increase the size of the data. As a result, the model may become slower and less accurate because important words get mixed with irrelevant ones.

---

### 3. Scenarios where removing stopwords is harmful

Although stopword removal is useful, in some cases it can change the meaning of a sentence:

- In **sentiment analysis**, removing words like "not" can completely reverse the meaning.  
  Example: "I am not happy" → removing "not" makes it "I am happy"

- In **question answering systems**, words like "what", "when", "where" are very important to understand the intent of the question.

---

### 4. Stemming vs Lemmatization

**Stemming:**  
Stemming is a technique where words are reduced to their base form by simply cutting off prefixes or suffixes using basic rules. It does not consider the actual meaning of the word, so sometimes the result may not be a valid word.  
Example: "running" → "runn"  

Stemming is fast and simple, but it is not very accurate.

---

**Lemmatization:**  
Lemmatization is a more advanced technique that converts words to their base or root form using a dictionary and understanding of the word’s context. The output is always a meaningful and correct word.  
Example: "running" → "run"  

Lemmatization is slower than stemming, but it is more accurate and reliable.

##Task 2:Build Advanced Preprocessing Function

In [9]:
def preprocess_text(text):

    # check for empty or invalid input
    if not text or not isinstance(text, str):
        return [], ""

    # remove URLs and email patterns
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)

    # remove numbers from text
    text = re.sub(r'\d+', '', text)

    # handle repeated characters (e.g., sooo → so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # convert text to lowercase
    text = text.lower()

    # remove special characters and keep only alphabets
    text = re.sub(r'[^a-z\s]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # split into tokens (words)
    tokens = text.split()

    # remove short words except meaningful ones like 'no' and 'not'
    tokens = [w for w in tokens if len(w) > 2 or w in ["no", "not"]]

    # join tokens to form cleaned sentence
    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

##Task 3: Stress Testing

In [12]:
import pandas as pd
test_sentences = [
"Get 100% FREE access now!!!",
"I absolutely looooved this product 😍😍",
"Worst service ever... 0/10",
"Call me at 9876543210",
"This is THE best course!!!",
"Visit https://openai.com now!",
"Nooooo this is baaad!!!",
"OK OK OK I got it",
"Win $$$ now!!! Limited offer!!!",
"I am not happy with this"
]

results = []
# testing preprocessing function on different types of sentences
for text in test_sentences:
    tokens, cleaned = preprocess_text(text)
    results.append((text, tokens, cleaned))

print("The table above shows how raw noisy text is converted into clean tokens and sentences after preprocessing.")
df = pd.DataFrame(results, columns=["Original Text", "Tokens", "Cleaned Sentence"])
df

The table above shows how raw noisy text is converted into clean tokens and sentences after preprocessing.


,Original Text,Tokens,Cleaned Sentence
0,Get 100% FREE access now!!!,"[get, free, access, now]",get free access now
1,I absolutely looooved this product 😍😍,"[absolutely, loved, this, product]",absolutely loved this product
2,Worst service ever... 0/10,"[worst, service, ever]",worst service ever
3,Call me at 9876543210,[call],call
4,This is THE best course!!!,"[this, the, best, course]",this the best course
5,Visit https://openai.com now!,"[visit, now]",visit now
6,Nooooo this is baaad!!!,"[no, this, bad]",no this bad
7,OK OK OK I got it,[got],got
8,Win $$$ now!!! Limited offer!!!,"[win, now, limited, offer]",win now limited offer
9,I am not happy with this,"[not, happy, with, this]",not happy with this


##Task 4: Token Analytics


In [13]:
# function to calculate token statistics
def token_analysis(tokens):
    if len(tokens) == 0:
        return 0, 0, 0

    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / total

    return total, unique, round(avg_len, 2)

analysis_data = []

for text in test_sentences:
    tokens, _ = preprocess_text(text)
    total, unique, avg = token_analysis(tokens)
    analysis_data.append((text, total, unique, avg))

df_analysis = pd.DataFrame(analysis_data, columns=[
    "Sentence", "Total Tokens", "Unique Tokens", "Avg Length"
])

df_analysis

,Sentence,Total Tokens,Unique Tokens,Avg Length
0,Get 100% FREE access now!!!,4,4,4.00
1,I absolutely looooved this product 😍😍,4,4,6.50
2,Worst service ever... 0/10,3,3,5.33
3,Call me at 9876543210,1,1,4.00
4,This is THE best course!!!,4,4,4.25
5,Visit https://openai.com now!,2,2,4.00
6,Nooooo this is baaad!!!,3,3,3.00
7,OK OK OK I got it,1,1,3.00
8,Win $$$ now!!! Limited offer!!!,4,4,4.50
9,I am not happy with this,4,4,4.00


### Analysis

- Most noisy sentence: "Win $$$ now!!! Limited offer!!!"
- Most meaningful sentence: "I absolutely looooved this product 😍😍"

##Task 5: Frequency Analysis

In [14]:
from collections import Counter

all_tokens = []
# count frequency of each word
for text in test_sentences:
    tokens, _ = preprocess_text(text)
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Top 10 Frequent Words:", freq.most_common(10))
print("Least 5 Frequent Words:", freq.most_common()[-5:])

Top 10 Frequent Words: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5 Frequent Words: [('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


This shows the most commonly occurring words after preprocessing.

##Task 6: Build Full Pipeline

In [15]:
# full pipeline to process list of sentences
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(cleaned)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

output = full_pipeline(test_sentences)
output

{'tokens': ['get',
  'free',
  'access',
  'now',
  'absolutely',
  'loved',
  'this',
  'product',
  'worst',
  'service',
  'ever',
  'call',
  'this',
  'the',
  'best',
  'course',
  'visit',
  'now',
  'no',
  'this',
  'bad',
  'got',
  'win',
  'now',
  'limited',
  'offer',
  'not',
  'happy',
  'with',
  'this'],
 'clean_sentences': ['get free access now',
  'absolutely loved this product',
  'worst service ever',
  'call',
  'this the best course',
  'visit now',
  'no this bad',
  'got',
  'win now limited offer',
  'not happy with this']}

##Task 7: Error Handling

In [16]:
# Edge Cases
print(preprocess_text(""))          # Empty
print(preprocess_text("😍😍😍"))    # Only emoji
print(preprocess_text("123456"))    # Only numbers

([], '')
([], '')
([], '')


## Conclusion

This project demonstrates a robust NLP preprocessing pipeline capable of handling noisy real-world text such as emojis, URLs, and repeated characters.

The processed output is clean and suitable for machine learning models.